In [77]:
!pip install biopython
!pip install transformers torch

In [78]:
import requests
from Bio import SeqIO
from io import StringIO
import torch
import itertools
from transformers import AutoTokenizer, EsmForMaskedLM

# Carichiamo il modello (usiamo l'8M per velocità)
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [79]:
def fasta_to_clean_string(file_path):
    sequences = []
    # parse gestisce file con una o più proteine
    for record in SeqIO.parse(file_path, "fasta"):
        # record.seq è un oggetto speciale, lo trasformiamo in stringa semplice
        clean_seq = str(record.seq)
        sequences.append(clean_seq)

    return sequences

In [80]:
def importProtein(uniprot_id):

  url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
  response = requests.get(url)

  if response.status_code == 200:
      print(response.text) # Hai la sequenza pronta per l'AI
      print(fasta_to_clean_string(StringIO(response.text)))
      print('lunghezza',len(fasta_to_clean_string(StringIO(response.text))[0]))
      return fasta_to_clean_string(StringIO(response.text))
  else:
    print('nessuna proteina trovata')

In [81]:
def trova_punti_flessibili(sequenza):
    inputs = tokenizer(sequenza, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)
    tokens = inputs["input_ids"][0]

    punteggi = []
    # Saltiamo <cls> (0) e <eos> (ultimo)
    for i in range(1, len(tokens) - 1):
        token_id = tokens[i]
        prob = probs[0, i, token_id].item()
        punteggi.append((i-1, sequenza[i-1], prob)) # (indice, amminoacido, probabilità)

    # Restituiamo la lista ordinata dal più "scarso" al più "sicuro"
    return sorted(punteggi, key=lambda x: x[2])

In [82]:
def maschera_flessibili(sequenza, soglia=None, max_maschere=None):
    punteggi=trova_punti_flessibili(sequenza)
    seq_list = list(sequenza)
    posizioni_da_mascherare = []

    for idx, aa, prob in punteggi:
        # Se superiamo il numero massimo di maschere, ci fermiamo
        if max_maschere and len(posizioni_da_mascherare) >= max_maschere:
            break
        # Se impostiamo una soglia (es. 0.2) e la prob è più alta, ci fermiamo
        if soglia and prob > soglia:
            break

        posizioni_da_mascherare.append(idx)

    for pos in posizioni_da_mascherare:
        seq_list[pos] = tokenizer.mask_token

    return "".join(seq_list), posizioni_da_mascherare

In [83]:
def genera_nuove_sequenze(sequenza_mascherata, n_top=5):
    inputs = tokenizer(sequenza_mascherata, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits

    mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

    opzioni_per_posizione = []
    for idx in mask_indices:
        # Prendiamo i top amminoacidi per ogni maschera
        probs = torch.softmax(logits[0, idx], dim=-1)
        top_probs, top_tokens = torch.topk(probs, 5) # Esaminiamo i primi 5 per ogni buco

        candidati = []
        for t in top_tokens:
            aa = tokenizer.decode([t])
            if len(aa) == 1 and aa.isalpha():
                candidati.append(aa)
        opzioni_per_posizione.append(candidati)

    # Creiamo tutte le combinazioni possibili tra i top amminoacidi
    combinazioni = list(itertools.product(*opzioni_per_posizione))

    nuove_sequenze = []
    for combo in combinazioni[:n_top]:
        temp_seq = sequenza_mascherata
        for aa in combo:
            temp_seq = temp_seq.replace(tokenizer.mask_token, aa, 1)
        nuove_sequenze.append(temp_seq)

    return nuove_sequenze

In [84]:
StartingProtein=input()
sequence=importProtein(StartingProtein)[0]
print(trova_punti_flessibili(sequence))
masked_seq, pos_mask=maschera_flessibili(sequence,soglia=0.01)
print("seq",masked_seq)
print("pos",pos_mask)
print("nuove seq"genera_nuove_sequenze(masked_seq))

P11006
>sp|P11006|MAGA_XENLA Magainins OS=Xenopus laevis OX=8355 GN=magainins PE=1 SV=1
MFKGLFICSLIAVICANALPQPEASADEDMDEREVRGIGKFLHSAGKFGKAFVGEIMKSK
RDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADE
DLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDLDEREVRGIGKFL
HSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDFDEREVRGIGKFLHSAKKFGKAFVGEI
MNSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVDDRR
WVE

['MFKGLFICSLIAVICANALPQPEASADEDMDEREVRGIGKFLHSAGKFGKAFVGEIMKSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDFDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVDDRRWVE']
lunghezza 303
[(45, 'G', 0.01797887682914734), (57, 'K', 0.061800092458724976), (213, 'F', 0.19217437505722046), (3, 'G', 0.1994011402130127), (300, 'W', 0.23719333112239838), (14, 'C', 0.31848955154418945), (297, 'D', 0.35491523146629333), (7, 'C', 0.36523857712745667), (29, 'M', 0.3662731051